In [2]:
import pandas as pd
import kagglehub
import os

path = kagglehub.competition_download('titanic')
train = pd.read_csv(os.path.join(path, 'train.csv'))
test = pd.read_csv(os.path.join(path, 'test.csv'))

print(train.shape, test.shape)

(891, 12) (418, 11)


In [3]:
full = pd.concat([train, test], sort=False, ignore_index=True)
full.isnull().sum()

PassengerId       0
Survived        418
Pclass            0
Name              0
Sex               0
Age             263
SibSp             0
Parch             0
Ticket            0
Fare              1
Cabin          1014
Embarked          2
dtype: int64

In [4]:
full['Title'] = full['Name'].str.extract(r',\s*([^.]*)\.')
title_map = {
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
    'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
    'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare'
}
full['Title'] = full['Title'].replace(title_map)
full.loc[~full['Title'].isin(['Mr', 'Miss', 'Mrs', 'Master', 'Rare']), 'Title'] = 'Rare'

In [5]:
full['FamilySize'] = full['SibSp'] + full['Parch'] + 1
full['IsAlone'] = (full['FamilySize'] == 1).astype(int)

In [6]:
# Age: median within Title+Pclass group instead of one global median
full['Age'] = full.groupby(['Title', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
full['Age'] = full['Age'].fillna(full['Age'].median())

# Fare: median within Pclass, then log-transform (fare is heavily right-skewed)
full['Fare'] = full.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
full['FareLog'] = np.log1p(full['Fare'])

# Embarked
full['Embarked'] = full['Embarked'].fillna(full['Embarked'].mode()[0])

In [7]:
# Deck: first letter of Cabin, missing cabin is its own category (predictive on its own)
full['Deck'] = full['Cabin'].str[0].fillna('M')

# Age bins
full['AgeBin'] = pd.cut(full['Age'], bins=[0, 12, 18, 35, 60, 100], labels=[0, 1, 2, 3, 4]).astype(int)

# Ticket group size: passengers sharing a ticket number often shared a fate
ticket_counts = full['Ticket'].value_counts()
full['TicketGroup'] = full['Ticket'].map(ticket_counts)

In [8]:
full['Sex'] = full['Sex'].map({'male': 0, 'female': 1})
full = pd.get_dummies(full, columns=['Title', 'Embarked', 'Deck'], drop_first=False)

In [9]:
feature_cols = [c for c in full.columns if c not in
                ['PassengerId', 'Survived', 'Name', 'Ticket', 'Cabin', 'Fare']]

train_fe = full.iloc[:len(train)].copy()
test_fe = full.iloc[len(train):].copy()

X = train_fe[feature_cols]
y = train_fe['Survived'].astype(int)
X_test = test_fe[feature_cols]

In [10]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

rf = RandomForestClassifier(n_estimators=500, max_depth=6, min_samples_leaf=2, random_state=42)
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
lr = LogisticRegression(C=0.5, max_iter=1000, random_state=42)

ensemble = VotingClassifier(estimators=[('rf', rf), ('gb', gb), ('lr', lr)], voting='soft')

scores = cross_val_score(ensemble, X, y, cv=cv, scoring='accuracy')
print(f"Ensemble CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

Ensemble CV accuracy: 0.8350 (+/- 0.0282)


In [11]:
ensemble.fit(X, y)
predictions = ensemble.predict(X_test)

output = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': predictions.astype(int)})
output.to_csv('submission.csv', index=False)
print("Saved!")

Saved!
